# Regulatory coupling and communication directionality

Run this notebook from top to bottom to process ageing mouse brain and HGSOC.
It loads saved SpiderNet and comparator results, builds NMF-LR factors once per study,
scores curated receiver targets and sender regulators, and evaluates reciprocal
edges and an edge-reversal control. The original-direction MI assignment is held
fixed for the reversal comparison. Two combined PDF/PNG displays use the saved
per-study tables.

See [README.md](README.md) for upstream inputs, manuscript mapping, execution
order and the command to redraw the figures from existing tables.


## 0. User controls

A normal run analyzes both studies sequentially, using an isolated kernel for
the second study. `SPIDERNET_SINGLE_DATASET` selects one study. NMF-LR is
rebuilt once per study with the original fixed settings. Set
`SPIDERNET_RECOMPUTE_NMFLR=0` to reuse archived `Factor_LR_list.pkl` files
known to match these settings; use plot-only mode when only redrawing figures.


In [ ]:
import os
from pathlib import Path
from benchmark_config import BENCHMARK_DIR

DATASETS_TO_RUN = ["AgingMousebrain", "HGSOC"]
DATASET = os.environ.get("SPIDERNET_SINGLE_DATASET", DATASETS_TO_RUN[0])
RUN_ALL_STUDIES = "SPIDERNET_SINGLE_DATASET" not in os.environ
NOTEBOOK_PATH = Path(os.environ.get(
    "SPIDERNET_DIRECTIONALITY_NOTEBOOK",
    str(BENCHMARK_DIR / "CCC_Coupling_benchmark_with_directionality.ipynb"),
)).resolve()
RECOMPUTE_NMFLR = os.environ.get("SPIDERNET_RECOMPUTE_NMFLR", "1") == "1"
NMF_RANDOM_STATE = 0
NMF_MAX_ITER = 1000

# Directionality analysis controls
DIRECTIONALITY_MAX_SCATTER_POINTS = 0  # scatter sampling is unnecessary for the final figures
DIRECTIONALITY_RANDOM_SEED = 20260726
DIRECTIONALITY_SAVE_PAIRED_SCATTER_SAMPLE = False


## 1. Imports and environment setup

In [ ]:
import os
import json
import pickle

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from scipy.stats import spearmanr
from scipy.stats import t

import torch

def scatter_nanmean(src, index, dim=0, dim_size=None):
    """NaN-aware mean aggregation along dim 0 using native PyTorch."""
    if dim != 0 or src.ndim != 2 or index.ndim != 1:
        raise ValueError("scatter_nanmean expects 2D src, 1D index, and dim=0.")
    if dim_size is None:
        dim_size = int(index.max().item()) + 1 if index.numel() else 0
    valid = torch.isfinite(src)
    values = torch.where(valid, src, torch.zeros_like(src))
    sums = torch.zeros((dim_size, src.shape[1]), dtype=src.dtype, device=src.device)
    counts = torch.zeros_like(sums)
    sums.index_add_(0, index, values)
    counts.index_add_(0, index, valid.to(src.dtype))
    means = sums / counts.clamp_min(1)
    return means.masked_fill(counts.eq(0), float("nan"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

from benchmark_common import (
    build_nmflr_factors,
    load_commot_outputs,
    load_sccchain_outputs,
    load_spacia_outputs,
    dedup_gene_sets,
    cell_by_pair_mean_genesets,
    compute_rowmax_spearman_df,
    mean_ci95,
    make_benchmark1_tables,
)


## 2. Dataset-specific configuration

Define all dataset-dependent paths and metadata fields

In [ ]:
from benchmark_config import DATASET_CONFIG

if DATASET not in DATASET_CONFIG:
    raise ValueError(f"Unsupported DATASET: {DATASET}")
cfg = DATASET_CONFIG[DATASET].copy()
cfg


In [ ]:
# Define SpiderNet result, processed-data, and benchmark output paths
save_path_main = cfg["results_path_main"]
save_path_main_cur = os.path.join(save_path_main, cfg["version"])
os.makedirs(save_path_main_cur, exist_ok=True)

result_tag = f"SpiderNet_Result_dim{cfg['dim_envir']}"

file_savepath_main = os.path.join(save_path_main_cur, result_tag)
if not file_savepath_main.endswith(os.sep):
    file_savepath_main += os.sep
file_savepath_main = file_savepath_main.replace("\\", "/")

processed_data_dir_cfg = cfg.get("processed_data_dir", None)
processed_data_candidates = []

if processed_data_dir_cfg is not None and str(processed_data_dir_cfg).strip() != "":
    processed_data_candidates.append(str(processed_data_dir_cfg))
processed_data_candidates.extend([
    os.path.join(save_path_main, "ProcessedData"),
    os.path.join(save_path_main_cur, "ProcessedData"),
])

processed_data_dir = None
for candidate in processed_data_candidates:
    candidate_norm = candidate.replace("\\", "/")
    if os.path.exists(candidate_norm):
        processed_data_dir = candidate_norm
        break

if processed_data_dir is None:
    processed_data_dir = processed_data_candidates[0].replace("\\", "/")

benchmark_outdir = os.path.join(file_savepath_main, "Unified_CCC_benchmarking").replace("\\", "/")
os.makedirs(benchmark_outdir, exist_ok=True)

directionality_outdir = os.path.join(
    benchmark_outdir,
    "Directionality_analyses",
).replace("\\", "/")
os.makedirs(directionality_outdir, exist_ok=True)

print("SpiderNet result directory:", file_savepath_main)
print("Processed data directory:", processed_data_dir)
print("Benchmark output directory:", benchmark_outdir)
print("Directionality output directory:", directionality_outdir)


## 3. Load SpiderNet outputs

Read the precomputed SpiderNet objects needed for benchmarking, including AnnData objects, graph data, and edge-level meta-interaction features.

In [ ]:
required_processed_files = [
    "adata_list.pkl",
    "SpiderNet_data_pyg_list.pkl",
    "LR_list.pkl",
]

required_result_files = [
    "Factor_envir_list.pkl",
]

missing_processed = [
    file_name for file_name in required_processed_files
    if not os.path.exists(os.path.join(processed_data_dir, file_name))
]
missing_result = [
    file_name for file_name in required_result_files
    if not os.path.exists(os.path.join(file_savepath_main, file_name))
]

if missing_processed:
    raise FileNotFoundError(
        "ProcessedData is incomplete. Missing files under "
        f"{processed_data_dir}: {missing_processed}\n"
        "If your ProcessedData folder is stored outside the version directory, "
        "set cfg['processed_data_dir'] explicitly or place it at the dataset-level "
        "Results folder."
    )

if missing_result:
    raise FileNotFoundError(
        "SpiderNet result directory is incomplete. Missing files under "
        f"{file_savepath_main}: {missing_result}"
    )

adata_list_path = os.path.join(processed_data_dir, "adata_list.pkl")
SpiderNet_data_pyg_list_path = os.path.join(processed_data_dir, "SpiderNet_data_pyg_list.pkl")
LR_list_path = os.path.join(processed_data_dir, "LR_list.pkl")

Factor_envir_list_path = os.path.join(file_savepath_main, "Factor_envir_list.pkl")
Factor_LR_list_path = os.path.join(file_savepath_main, "Factor_LR_list.pkl")

adata_all_path = os.path.join(processed_data_dir, "adata_all.h5ad")

adata_list = pd.read_pickle(adata_list_path)
SpiderNet_data_pyg_list = pd.read_pickle(SpiderNet_data_pyg_list_path)
Factor_envir_list = pd.read_pickle(Factor_envir_list_path)

if os.path.exists(adata_all_path):
    adata_all = sc.read_h5ad(adata_all_path)
else:
    if len(adata_list) == 0:
        raise ValueError(f"No slices found in {adata_list_path}")
    if len(adata_list) == 1:
        adata_all = adata_list[0].copy()
    else:
        adata_all = sc.concat(adata_list, join="inner", merge="same")

print("n_slices:", len(adata_list))
print("adata_all:", adata_all.shape)
print("cell class column:", cfg["cellclass_name"])
print("Processed adata_list path:", adata_list_path)
print("Processed SpiderNet graph path:", SpiderNet_data_pyg_list_path)
print("SpiderNet factor path:", Factor_envir_list_path)
print("NMF-LR factor cache path:", Factor_LR_list_path)


### NMF-LR

In [ ]:
if (not RECOMPUTE_NMFLR) and os.path.exists(Factor_LR_list_path):
    Factor_LR_list = pd.read_pickle(Factor_LR_list_path)
    print("Loaded existing NMF-LR factors:", Factor_LR_list_path)
else:
    Factor_LR_list = build_nmflr_factors(
        SpiderNet_data_pyg_list,
        n_components=cfg["dim_envir"],
        random_state=NMF_RANDOM_STATE,
        max_iter=NMF_MAX_ITER,
    )
    with open(Factor_LR_list_path, "wb") as f:
        pickle.dump(Factor_LR_list, f)
    print("Saved NMF-LR factors:", Factor_LR_list_path)


## 4. Aggregation and feature labels

Shared loading and scoring functions are imported from `benchmark_common.py`.
The finite-value PyTorch aggregation and LR-label handling below define this
notebook's communication profiles.


In [ ]:
METHOD_COLORS = {
    "SpiderNet": {"edge": "#9F3B38", "fill": "#E1B6A7"},
    "SpiderNet-reverse direction": {"edge": "#FF85BB", "fill": "#FFCEE3"},
    "NMF-LR": {"edge": "#82CCE2", "fill": "#D4ECF1"},
    "COMMOT": {"edge": "#519384", "fill": "#B9CEC7"},
    "ScCChain": {"edge": "#636491", "fill": "#A6A2B9"},
    "Spacia": {"edge": "#FED881", "fill": "#FFF2D2"},
}


def join_plus(x):
    if isinstance(x, str):
        return x
    return "+".join(str(i) for i in x if i is not None and str(i) != "nan")


def as_gene_list(x):
    if isinstance(x, str):
        return [x]
    return [str(g) for g in x if g is not None and str(g) != "nan"]




def aggregate_edge_features(edge_features, edge_index_col, num_cells, device):
    agg = scatter_nanmean(
        torch.as_tensor(edge_features, dtype=torch.float32, device=device),
        edge_index_col.to(torch.int64).to(device),
        dim=0,
        dim_size=num_cells,
    ).to("cpu").numpy()
    return agg


def aggregate_sender_receiver(edge_features, edge_index, num_cells, device):
    sender_agg = aggregate_edge_features(edge_features, edge_index[:, 0], num_cells, device)
    receiver_agg = aggregate_edge_features(edge_features, edge_index[:, 1], num_cells, device)
    return {
        "sender": sender_agg,
        "receiver": receiver_agg,
        "combined": np.hstack([sender_agg, receiver_agg]),
    }


## 5. Load comparator outputs

Load the communication features produced by the comparator methods and align them to the same edge structure used by SpiderNet.

In [ ]:
COMMOT_score_list = load_commot_outputs(
    adata_list,
    SpiderNet_data_pyg_list,
    cfg,
)

ScCChain_CPscore_list = load_sccchain_outputs(
    adata_list,
    SpiderNet_data_pyg_list,
    cfg,
)

Spacia_CP_list = load_spacia_outputs(
    adata_list,
    SpiderNet_data_pyg_list,
    cfg,
)

print("COMMOT slices:", len(COMMOT_score_list))
print("ScCChain slices:", len(ScCChain_CPscore_list))
print("Spacia enabled:", cfg["include_spacia"])


# Benchmark 1. Curated gene-coupling benchmark

This benchmark asks whether inferred communication features recover curated biology linked to ligand-receptor pairs.

For each slice:
1. Build a **cell × LR-set** matrix from curated **receiver target genes** and **sender upstream regulators**.
2. Aggregate each method's edge-level outputs to the **receiver side** for target analysis and the **sender side** for regulator analysis.
3. For each representative LR set, compute the **maximum Spearman correlation** between curated gene activity and inferred communication features.
4. Summarize the distribution of correlations across LR sets and slices.

Higher correlation indicates that a method more strongly captures curated downstream targets or upstream regulators associated with the communication axis.


In [ ]:
# Load ligand-receptor pairs and curated target-gene annotations
gene_names = adata_all.var_names.tolist()
LR_list = pd.read_pickle(os.path.join(processed_data_dir, "LR_list.pkl"))

LR_merge = [f"{join_plus(LR_sub[0])}->{join_plus(LR_sub[1])}" for LR_sub in LR_list]

with open(os.path.join(benchmark_outdir, "gene_names.txt"), "w") as f:
    for gene in gene_names:
        f.write(f"{gene}\n")

with open(os.path.join(benchmark_outdir, "LR_pairs.txt"), "w") as f:
    for lr in LR_merge:
        f.write(f"{lr}\n")

with open(os.path.join(file_savepath_main, cfg["targets_json_name"]), "r") as f:
    targets_by_LR = json.load(f)

print("n_genes:", len(gene_names))
print("n_LR_pairs:", len(LR_list))
print("n_target_annotated_pairs:", len(targets_by_LR))


In [ ]:
targets_by_setid, repkey_targets, setid_by_lr_targets, lrs_by_setid_targets = dedup_gene_sets(targets_by_LR)
targets_by_LR_unique = {
    repkey_targets[set_id]: genes
    for set_id, genes in targets_by_setid.items()
}

print("target-annotated LR keys before deduplication:", len(targets_by_LR))
print("unique target sets:", len(targets_by_LR_unique))


In [ ]:
# Load curated upstream regulator annotations from OmniPath
interactions_regulatory = pd.read_csv(cfg["omnipath_regulatory_csv"])
interactions_regulatory_stimulation = interactions_regulatory[
    interactions_regulatory["is_stimulation"] & (~interactions_regulatory["is_inhibition"])
]

upstream_regulators_dict = {}
for LR in LR_list:
    ligand_genes = as_gene_list(LR[0])
    LR_key = f"{join_plus(LR[0])}->{join_plus(LR[1])}"

    regulators = interactions_regulatory_stimulation.loc[
        interactions_regulatory_stimulation["target_genesymbol"].isin(ligand_genes),
        "source_genesymbol",
    ].dropna().astype(str).unique().tolist()
    regulators = [r for r in regulators if r in gene_names]

    if len(regulators) >= cfg["min_upstream_regulators"]:
        upstream_regulators_dict[LR_key] = regulators

regulators_by_setid, repkey_regulators, setid_by_lr_regulators, lrs_by_setid_regulators = dedup_gene_sets(
    upstream_regulators_dict
)
regulators_by_LR_unique = {
    repkey_regulators[set_id]: genes
    for set_id, genes in regulators_by_setid.items()
}

print("annotated regulator LR keys:", len(upstream_regulators_dict))
print("unique regulator sets:", len(regulators_by_LR_unique))


In [ ]:
# Run Benchmark 1 for all slices and methods
target_corr = {method: [] for method in ["SpiderNet", "NMF-LR", "COMMOT", "ScCChain"]}
regulator_corr = {method: [] for method in ["SpiderNet", "NMF-LR", "COMMOT", "ScCChain"]}

if cfg["include_spacia"]:
    target_corr["Spacia"] = []
    regulator_corr["Spacia"] = []

for slice_index in range(len(adata_list)):
    print(f"Benchmark 1 | slice {slice_index + 1}/{len(adata_list)}")
    adata_sub = adata_list[slice_index]
    edge_index_sub = SpiderNet_data_pyg_list[slice_index]["edge_index"]
    num_cell_sub = SpiderNet_data_pyg_list[slice_index].x.shape[0]

    target_df, _, _ = cell_by_pair_mean_genesets(
        adata_sub,
        targets_by_LR_unique,
        use_layer=None,
        fill_value=np.nan,
    )
    regulator_df, _, _ = cell_by_pair_mean_genesets(
        adata_sub,
        regulators_by_LR_unique,
        use_layer=None,
        fill_value=np.nan,
    )

    method_edge_features = {
        "SpiderNet": np.asarray(Factor_envir_list[slice_index], dtype=float),
        "NMF-LR": np.asarray(Factor_LR_list[slice_index], dtype=float),
        "COMMOT": np.asarray(COMMOT_score_list[slice_index], dtype=float),
        "ScCChain": np.asarray(ScCChain_CPscore_list[slice_index], dtype=float),
    }
    if cfg["include_spacia"]:
        method_edge_features["Spacia"] = np.asarray(Spacia_CP_list[slice_index], dtype=float)

    for method_name, edge_features in method_edge_features.items():
        agg = aggregate_sender_receiver(edge_features, edge_index_sub, num_cell_sub, device)

        target_corr[method_name].append(
            compute_rowmax_spearman_df(target_df, agg["receiver"], slice_index)
        )
        regulator_corr[method_name].append(
            compute_rowmax_spearman_df(regulator_df, agg["sender"], slice_index)
        )

target_corr_frames = {k: pd.concat(v, axis=0) for k, v in target_corr.items()}
regulator_corr_frames = {k: pd.concat(v, axis=0) for k, v in regulator_corr.items()}

for name, df in target_corr_frames.items():
    df.to_csv(os.path.join(benchmark_outdir, f"Benchmark1_target_corr_{name.replace('/', '_')}.csv"))
for name, df in regulator_corr_frames.items():
    df.to_csv(os.path.join(benchmark_outdir, f"Benchmark1_regulator_corr_{name.replace('/', '_')}.csv"))

print("Benchmark 1 finished.")


In [ ]:
benchmark1_long_df, benchmark1_plot_summary_df, benchmark1_stats_df, benchmark1_method_order = make_benchmark1_tables(
    target_corr_frames,
    regulator_corr_frames,
)

benchmark1_long_df.to_csv(os.path.join(benchmark_outdir, "Benchmark1_long.csv"), index=False)
benchmark1_plot_summary_df.to_csv(os.path.join(benchmark_outdir, "Benchmark1_plot_summary.csv"), index=False)
benchmark1_stats_df.to_csv(os.path.join(benchmark_outdir, "Benchmark1_pairwise_stats.csv"), index=False)

print("Saved Benchmark 1 tables:")
print(" -", os.path.join(benchmark_outdir, "Benchmark1_long.csv"))
print(" -", os.path.join(benchmark_outdir, "Benchmark1_plot_summary.csv"))
print(" -", os.path.join(benchmark_outdir, "Benchmark1_pairwise_stats.csv"))

print("\nBenchmark 1 plot summary (first 12 rows):")

print("\nBenchmark 1 pairwise statistics (first 12 rows):")
# Intermediate table display omitted.


# Directionality module

This module is shared by the HGSOC and ageing mouse brain datasets selected in `DATASET`.

**Analysis 1** pairs reciprocal directed edges and quantifies the Pearson correlation between MI activity assigned to `cell 1 → cell 2` and `cell 2 → cell 1`.

**Analysis 2** preserves each fitted MI activity but swaps sender and receiver endpoints. Curated sender-regulator and receiver-target coupling is then evaluated at the MI selected from the original SpiderNet direction. This prevents the reversed control from selecting a different MI post hoc.

The module also produces an additional Benchmark-1-style plot containing all comparator methods plus `SpiderNet-reverse direction` as the final method.

In [ ]:
# Directionality-specific shared helpers
REVERSE_DIRECTION_METHOD = "SpiderNet-reverse direction"
DIRECTIONALITY_METHOD_ORDER = ["SpiderNet", REVERSE_DIRECTION_METHOD]


def dir_to_numpy_2d(x, name="array"):
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    elif sp.issparse(x):
        x = x.toarray()
    else:
        x = np.asarray(x)

    if x.ndim != 2:
        raise ValueError(f"{name} must be 2D; got shape {x.shape}")
    return x


def dir_edge_index_to_numpy(edge_index):
    edge_index = dir_to_numpy_2d(edge_index, name="edge_index")
    if edge_index.shape[1] == 2:
        out = edge_index
    elif edge_index.shape[0] == 2:
        out = edge_index.T
    else:
        raise ValueError(
            "edge_index must have shape E x 2 or 2 x E; "
            f"got {edge_index.shape}"
        )
    return out.astype(np.int64, copy=False)




def dir_bh_fdr(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    qvalues = np.full(pvalues.shape, np.nan, dtype=float)
    finite_idx = np.where(np.isfinite(pvalues))[0]
    if finite_idx.size == 0:
        return qvalues

    p = pvalues[finite_idx]
    order = np.argsort(p)
    ranked = p[order]
    m = ranked.size
    adjusted = ranked * m / np.arange(1, m + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)

    q_ordered = np.empty_like(adjusted)
    q_ordered[order] = adjusted
    qvalues[finite_idx] = q_ordered
    return qvalues




def dir_get_slice_label_and_metadata(adata_sub, slice_index):
    """Use the configured sample/age field when it is constant within a slice."""
    base_label = f"Slice_{slice_index + 1}"
    obs_key = cfg.get("sample_obs_key")
    metadata_value = np.nan

    if obs_key is None or obs_key not in adata_sub.obs.columns:
        return base_label, obs_key, metadata_value

    values = pd.Series(adata_sub.obs[obs_key]).dropna().unique().tolist()
    if len(values) != 1:
        return base_label, obs_key, metadata_value

    metadata_value = values[0]
    safe_value = str(metadata_value).replace(" ", "_").replace("/", "-")
    return f"{base_label}_{obs_key}_{safe_value}", obs_key, metadata_value


def dir_collapse_duplicate_directed_edges(edge_index, edge_features, n_cells):
    """Return unique directed edges and feature-wise mean activity per edge."""
    edge_index = dir_edge_index_to_numpy(edge_index)
    edge_features = dir_to_numpy_2d(edge_features, name="edge_features").astype(float, copy=False)

    if edge_index.shape[0] != edge_features.shape[0]:
        raise ValueError(
            "edge_index and edge_features must contain the same number of edges; "
            f"got {edge_index.shape[0]} and {edge_features.shape[0]}"
        )

    src = edge_index[:, 0]
    dst = edge_index[:, 1]
    if np.any(src < 0) or np.any(dst < 0) or np.any(src >= n_cells) or np.any(dst >= n_cells):
        raise IndexError("edge_index contains node IDs outside [0, n_cells).")

    nonself = src != dst
    src = src[nonself]
    dst = dst[nonself]
    edge_features = edge_features[nonself]

    directed_keys = src.astype(np.int64) * np.int64(n_cells) + dst.astype(np.int64)
    unique_keys, unique_first_index, inverse, duplicate_counts = np.unique(
        directed_keys,
        return_index=True,
        return_inverse=True,
        return_counts=True,
    )

    if unique_keys.size == directed_keys.size:
        unique_features = edge_features[unique_first_index]
    else:
        n_unique = unique_keys.size
        n_features = edge_features.shape[1]
        unique_features = np.full((n_unique, n_features), np.nan, dtype=float)
        for j in range(n_features):
            vals = edge_features[:, j]
            finite = np.isfinite(vals)
            sums = np.bincount(inverse[finite], weights=vals[finite], minlength=n_unique)
            counts = np.bincount(inverse[finite], minlength=n_unique)
            valid = counts > 0
            unique_features[valid, j] = sums[valid] / counts[valid]

    unique_src = unique_keys // np.int64(n_cells)
    unique_dst = unique_keys % np.int64(n_cells)
    return unique_keys, unique_src, unique_dst, unique_features, duplicate_counts


def dir_extract_reciprocal_pairs(edge_index, edge_features, n_cells):
    """Pair MI(u→v) with MI(v→u), counting each unordered pair once."""
    keys, src, dst, features, duplicate_counts = dir_collapse_duplicate_directed_edges(
        edge_index, edge_features, n_cells
    )

    reverse_keys = dst.astype(np.int64) * np.int64(n_cells) + src.astype(np.int64)
    reverse_pos = np.searchsorted(keys, reverse_keys)
    in_bounds = reverse_pos < keys.size
    has_reverse = np.zeros(keys.size, dtype=bool)
    has_reverse[in_bounds] = keys[reverse_pos[in_bounds]] == reverse_keys[in_bounds]

    canonical = has_reverse & (src < dst)
    forward_pos = np.where(canonical)[0]
    reverse_pos = reverse_pos[canonical]

    pair_meta = pd.DataFrame({
        "cell1": src[forward_pos].astype(int),
        "cell2": dst[forward_pos].astype(int),
    })
    forward_activity = features[forward_pos]
    reverse_activity = features[reverse_pos]

    diagnostics = {
        "n_input_edges": int(dir_edge_index_to_numpy(edge_index).shape[0]),
        "n_unique_nonself_directed_edges": int(keys.size),
        "n_duplicate_directed_edge_rows": int(np.sum(duplicate_counts - 1)),
        "n_reciprocal_unordered_pairs": int(forward_pos.size),
    }
    return pair_meta, forward_activity, reverse_activity, diagnostics


def dir_initialize_pearson_accumulator(n_features):
    return {
        "n": np.zeros(n_features, dtype=np.int64),
        "sum_x": np.zeros(n_features, dtype=np.float64),
        "sum_y": np.zeros(n_features, dtype=np.float64),
        "sum_xx": np.zeros(n_features, dtype=np.float64),
        "sum_yy": np.zeros(n_features, dtype=np.float64),
        "sum_xy": np.zeros(n_features, dtype=np.float64),
    }


def dir_update_pearson_accumulator(acc, x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if x.shape != y.shape:
        raise ValueError(f"x and y must have identical shapes; got {x.shape} and {y.shape}")

    for j in range(x.shape[1]):
        valid = np.isfinite(x[:, j]) & np.isfinite(y[:, j])
        xv = x[valid, j]
        yv = y[valid, j]
        acc["n"][j] += xv.size
        acc["sum_x"][j] += xv.sum(dtype=np.float64)
        acc["sum_y"][j] += yv.sum(dtype=np.float64)
        acc["sum_xx"][j] += np.dot(xv, xv)
        acc["sum_yy"][j] += np.dot(yv, yv)
        acc["sum_xy"][j] += np.dot(xv, yv)


def dir_finalize_pearson_accumulator(acc):
    n = acc["n"].astype(float)
    num = n * acc["sum_xy"] - acc["sum_x"] * acc["sum_y"]
    den_x = n * acc["sum_xx"] - acc["sum_x"] ** 2
    den_y = n * acc["sum_yy"] - acc["sum_y"] ** 2
    den = np.sqrt(np.maximum(den_x, 0) * np.maximum(den_y, 0))

    r = np.full(n.shape, np.nan, dtype=float)
    valid_r = (n >= 3) & (den > 0)
    r[valid_r] = num[valid_r] / den[valid_r]
    finite_clip = np.isfinite(r)
    r[finite_clip] = np.clip(r[finite_clip], -1, 1)

    p = np.full(n.shape, np.nan, dtype=float)
    finite_r = valid_r & np.isfinite(r)
    perfect = finite_r & (np.abs(r) >= 1)
    p[perfect] = 0.0
    regular = finite_r & (~perfect)
    t_stat = np.abs(r[regular]) * np.sqrt(
        (n[regular] - 2) / np.maximum(1 - r[regular] ** 2, np.finfo(float).tiny)
    )
    p[regular] = 2 * t.sf(t_stat, df=n[regular] - 2)

    slope = np.full(n.shape, np.nan, dtype=float)
    valid_slope = (n >= 2) & (den_x > 0)
    slope[valid_slope] = num[valid_slope] / den_x[valid_slope]
    mean_x = np.divide(acc["sum_x"], n, out=np.full(n.shape, np.nan), where=n > 0)
    mean_y = np.divide(acc["sum_y"], n, out=np.full(n.shape, np.nan), where=n > 0)
    intercept = mean_y - slope * mean_x

    return pd.DataFrame({
        "MI": [f"MI-{i + 1}" for i in range(len(n))],
        "MI_index_1based": np.arange(1, len(n) + 1),
        "n_pairs": acc["n"],
        "pearson_r": r,
        "p_value": p,
        "regression_slope_y_on_x": slope,
        "regression_intercept_y_on_x": intercept,
    })


def dir_update_priority_sample(sample, x, y, pair_meta, slice_index, rng, max_points):
    """Uniform row sample using independent random priorities."""
    n_rows = x.shape[0]
    if n_rows != len(pair_meta):
        raise ValueError("pair_meta rows must match paired activity rows.")
    if n_rows == 0 or max_points <= 0:
        return sample

    cell1 = pair_meta["cell1"].to_numpy(dtype=int)
    cell2 = pair_meta["cell2"].to_numpy(dtype=int)
    priorities = rng.random(n_rows)
    if n_rows > max_points:
        keep = np.argpartition(priorities, max_points - 1)[:max_points]
        priorities = priorities[keep]
        x = x[keep]
        y = y[keep]
        cell1 = cell1[keep]
        cell2 = cell2[keep]
        slice_ids = np.full(max_points, slice_index + 1, dtype=int)
    else:
        slice_ids = np.full(n_rows, slice_index + 1, dtype=int)

    if sample is None:
        combined = {
            "priority": priorities,
            "x": x.copy(),
            "y": y.copy(),
            "slice": slice_ids,
            "cell1": cell1,
            "cell2": cell2,
        }
    else:
        combined = {
            "priority": np.concatenate([sample["priority"], priorities]),
            "x": np.vstack([sample["x"], x]),
            "y": np.vstack([sample["y"], y]),
            "slice": np.concatenate([sample["slice"], slice_ids]),
            "cell1": np.concatenate([sample["cell1"], cell1]),
            "cell2": np.concatenate([sample["cell2"], cell2]),
        }

    if combined["priority"].size > max_points:
        keep = np.argpartition(combined["priority"], max_points - 1)[:max_points]
        combined = {key: value[keep] for key, value in combined.items()}

    return combined


## Directionality analysis 1. Reciprocal-edge MI activity correlation

In [ ]:
# Pair reciprocal edges and calculate pooled and slice-level Pearson correlations.
n_mis_directionality = dir_to_numpy_2d(
    Factor_envir_list[0], "Factor_envir_list[0]"
).shape[1]

if n_mis_directionality != cfg["dim_envir"]:
    print(
        f"Warning: configured dim_envir={cfg['dim_envir']}, "
        f"but loaded factors have {n_mis_directionality} dimensions."
    )

directionality_pearson_acc = dir_initialize_pearson_accumulator(n_mis_directionality)
directionality_rng = np.random.default_rng(DIRECTIONALITY_RANDOM_SEED)
directionality_scatter_sample = None
directionality_slice_summary_records = []
directionality_slice_corr_frames = []

for slice_index, (adata_sub, graph_sub, factor_sub) in enumerate(
    zip(adata_list, SpiderNet_data_pyg_list, Factor_envir_list)
):
    edge_index_sub = dir_edge_index_to_numpy(graph_sub["edge_index"])
    factor_sub = dir_to_numpy_2d(
        factor_sub,
        name=f"Factor_envir_list[{slice_index}]",
    )
    n_cells = int(adata_sub.n_obs)
    slice_label, metadata_key, metadata_value = dir_get_slice_label_and_metadata(
        adata_sub, slice_index
    )

    if factor_sub.shape[1] != n_mis_directionality:
        raise ValueError(
            f"Slice {slice_index + 1} has {factor_sub.shape[1]} MIs; "
            f"expected {n_mis_directionality}."
        )

    pair_meta, forward_activity, reverse_activity, diagnostics = dir_extract_reciprocal_pairs(
        edge_index_sub,
        factor_sub,
        n_cells=n_cells,
    )

    dir_update_pearson_accumulator(
        directionality_pearson_acc,
        forward_activity,
        reverse_activity,
    )

    slice_acc = dir_initialize_pearson_accumulator(n_mis_directionality)
    dir_update_pearson_accumulator(slice_acc, forward_activity, reverse_activity)
    slice_corr_df = dir_finalize_pearson_accumulator(slice_acc)
    slice_corr_df["p_fdr_bh"] = dir_bh_fdr(slice_corr_df["p_value"])
    slice_corr_df.insert(0, "Slice", slice_label)
    if metadata_key is not None:
        slice_corr_df.insert(1, metadata_key, metadata_value)
    directionality_slice_corr_frames.append(slice_corr_df)

    directionality_scatter_sample = dir_update_priority_sample(
        directionality_scatter_sample,
        forward_activity,
        reverse_activity,
        pair_meta,
        slice_index=slice_index,
        rng=directionality_rng,
        max_points=DIRECTIONALITY_MAX_SCATTER_POINTS,
    )

    diagnostics.update({
        "Slice": slice_label,
        "n_cells": n_cells,
    })
    if metadata_key is not None:
        diagnostics[metadata_key] = metadata_value
    directionality_slice_summary_records.append(diagnostics)

    n_finite_corr = int(np.isfinite(slice_corr_df["pearson_r"]).sum())
    print(
        f"{slice_label} ({slice_index + 1:02d}/{len(adata_list)}) | "
        f"reciprocal pairs = {diagnostics['n_reciprocal_unordered_pairs']:,} | "
        f"finite MI correlations = {n_finite_corr}/{n_mis_directionality}"
    )

directionality_reciprocal_slice_summary_df = pd.DataFrame(
    directionality_slice_summary_records
)
directionality_mi_corr_by_slice_df = pd.concat(
    directionality_slice_corr_frames,
    axis=0,
    ignore_index=True,
)
directionality_mi_corr_df = dir_finalize_pearson_accumulator(
    directionality_pearson_acc
)
directionality_mi_corr_df["p_fdr_bh"] = dir_bh_fdr(
    directionality_mi_corr_df["p_value"]
)

directionality_reciprocal_slice_summary_path = os.path.join(
    directionality_outdir,
    "Analysis1_reciprocal_edge_summary_by_slice.csv",
)
directionality_mi_corr_path = os.path.join(
    directionality_outdir,
    "Analysis1_MI_reverse_direction_Pearson.csv",
)
directionality_mi_corr_by_slice_path = os.path.join(
    directionality_outdir,
    "Analysis1_MI_reverse_direction_Pearson_by_slice.csv",
)

directionality_reciprocal_slice_summary_df.to_csv(
    directionality_reciprocal_slice_summary_path,
    index=False,
)
directionality_mi_corr_df.to_csv(
    directionality_mi_corr_path,
    index=False,
)
directionality_mi_corr_by_slice_df.to_csv(
    directionality_mi_corr_by_slice_path,
    index=False,
)

if DIRECTIONALITY_SAVE_PAIRED_SCATTER_SAMPLE and directionality_scatter_sample is not None:
    sample_df = pd.DataFrame({
        "Slice": directionality_scatter_sample["slice"],
        "cell1_local_index": directionality_scatter_sample["cell1"],
        "cell2_local_index": directionality_scatter_sample["cell2"],
    })
    for j in range(n_mis_directionality):
        sample_df[f"MI{j + 1}_cell1_to_cell2"] = directionality_scatter_sample["x"][:, j]
        sample_df[f"MI{j + 1}_cell2_to_cell1"] = directionality_scatter_sample["y"][:, j]
    sample_path = os.path.join(
        directionality_outdir,
        "Analysis1_reciprocal_pair_scatter_sample.csv.gz",
    )
    sample_df.to_csv(sample_path, index=False, compression="gzip")
    print("Saved scatter sample:", sample_path)

print("Saved:", directionality_reciprocal_slice_summary_path)
print("Saved:", directionality_mi_corr_path)
print("Saved:", directionality_mi_corr_by_slice_path)
# Intermediate table display omitted.


## Directionality analysis 2. Original versus edge-reversed regulatory coupling

In [ ]:
def dir_aggregate_edge_features_nanmean(edge_features, node_ids, n_cells):
    """Feature-wise mean of incident edge features for each node."""
    edge_features = dir_to_numpy_2d(
        edge_features,
        name="edge_features",
    ).astype(float, copy=False)
    node_ids = np.asarray(node_ids, dtype=np.int64)
    if edge_features.shape[0] != node_ids.size:
        raise ValueError("edge_features rows must match node_ids length.")

    n_features = edge_features.shape[1]
    sums = np.zeros((n_cells, n_features), dtype=np.float64)
    counts = np.zeros((n_cells, n_features), dtype=np.int64)
    finite = np.isfinite(edge_features)
    np.add.at(sums, node_ids, np.where(finite, edge_features, 0.0))
    np.add.at(counts, node_ids, finite.astype(np.int64))

    out = np.full((n_cells, n_features), np.nan, dtype=np.float64)
    np.divide(sums, counts, out=out, where=counts > 0)
    return out


def dir_aggregate_sender_receiver(edge_features, edge_index, n_cells):
    edge_index = dir_edge_index_to_numpy(edge_index)
    return {
        "sender": dir_aggregate_edge_features_nanmean(
            edge_features,
            edge_index[:, 0],
            n_cells,
        ),
        "receiver": dir_aggregate_edge_features_nanmean(
            edge_features,
            edge_index[:, 1],
            n_cells,
        ),
    }


def dir_compute_pair_mi_spearman_matrix(X_df, Y_array):
    """Return a curated-pair × MI Spearman correlation matrix."""
    X = np.asarray(X_df, dtype=float)
    Y = np.asarray(Y_array, dtype=float)

    if X.ndim != 2 or Y.ndim != 2:
        raise ValueError(f"X and Y must both be 2D, got {X.shape} and {Y.shape}")
    if X.shape[0] != Y.shape[0]:
        raise ValueError(f"X and Y must have equal n_cells, got {X.shape} and {Y.shape}")
    if X.shape[1] == 0:
        return np.empty((0, Y.shape[1]), dtype=float)
    if Y.shape[1] == 0:
        return np.full((X.shape[1], 0), np.nan, dtype=float)

    X = np.nan_to_num(X, nan=0.0)
    Y = np.nan_to_num(Y, nan=0.0)
    corr_all, _ = spearmanr(X, Y, axis=0)
    corr_all = np.asarray(corr_all, dtype=float)

    n_x = X.shape[1]
    n_y = Y.shape[1]
    expected = n_x + n_y
    if corr_all.ndim == 0:
        corr_all = np.array([[1.0, corr_all], [corr_all, 1.0]], dtype=float)
    if corr_all.shape != (expected, expected):
        raise ValueError(
            "Unexpected Spearman correlation-matrix shape: "
            f"got {corr_all.shape}, expected {(expected, expected)}"
        )
    return corr_all[:n_x, n_x:]


def dir_select_reference_argmax_mi(corr_pair_by_mi):
    """Select one MI per curated pair from original SpiderNet only."""
    corr_pair_by_mi = np.asarray(corr_pair_by_mi, dtype=float)
    selected = np.full(corr_pair_by_mi.shape[0], -1, dtype=int)
    for row_index, row in enumerate(corr_pair_by_mi):
        finite = np.isfinite(row)
        if finite.any():
            selected[row_index] = int(np.nanargmax(row))
    return selected


def dir_extract_selected_mi_correlations(
    corr_pair_by_mi,
    pair_names,
    selected_mi_indices,
    slice_label,
):
    """Extract correlations at a pre-specified LR-to-MI assignment."""
    corr_pair_by_mi = np.asarray(corr_pair_by_mi, dtype=float)
    selected_mi_indices = np.asarray(selected_mi_indices, dtype=int)
    pair_names = list(pair_names)

    if corr_pair_by_mi.shape[0] != len(pair_names):
        raise ValueError("The correlation-matrix rows must match pair_names.")
    if selected_mi_indices.size != len(pair_names):
        raise ValueError("selected_mi_indices must contain one entry per curated pair.")

    values = np.full(len(pair_names), np.nan, dtype=float)
    valid = (
        (selected_mi_indices >= 0)
        & (selected_mi_indices < corr_pair_by_mi.shape[1])
    )
    row_indices = np.arange(len(pair_names), dtype=int)
    values[valid] = corr_pair_by_mi[
        row_indices[valid],
        selected_mi_indices[valid],
    ]

    return pd.DataFrame(
        [values],
        index=[str(slice_label)],
        columns=pair_names,
    )


def dir_build_lr_mi_selection_records(
    pair_names,
    selected_mi_indices,
    reference_corr_matrix,
    reverse_corr_matrix,
    slice_label,
    metadata_key,
    metadata_value,
    feature_type,
):
    records = []
    for row_index, (pair_name, mi_index) in enumerate(
        zip(pair_names, selected_mi_indices)
    ):
        valid = 0 <= mi_index < reference_corr_matrix.shape[1]
        record = {
            "Slice": str(slice_label),
            "FeatureType": feature_type,
            "RepresentativeLR": pair_name,
            "Selected_MI_index_0based": int(mi_index) if valid else np.nan,
            "Selected_MI_index_1based": int(mi_index + 1) if valid else np.nan,
            "Selected_MI": f"MI-{mi_index + 1}" if valid else None,
            "SpiderNet_correlation": (
                float(reference_corr_matrix[row_index, mi_index]) if valid else np.nan
            ),
            "SpiderNet_reverse_direction_correlation_at_same_MI": (
                float(reverse_corr_matrix[row_index, mi_index]) if valid else np.nan
            ),
        }
        if metadata_key is not None:
            record[metadata_key] = metadata_value
        records.append(record)
    return records


def dir_corr_frames_to_long(frame_dict, feature_type):
    records = []
    for method_name, df in frame_dict.items():
        tmp = df.copy()
        tmp["Slice"] = tmp.index.astype(str)
        tmp = tmp.melt(
            id_vars=["Slice"],
            var_name="RepresentativeLR",
            value_name="Correlation",
        )
        tmp["Method"] = method_name
        tmp["FeatureType"] = feature_type
        records.append(tmp)

    out = pd.concat(records, axis=0, ignore_index=True)
    out["Correlation"] = pd.to_numeric(out["Correlation"], errors="coerce")
    return out


In [ ]:
directionality_target_corr = {
    method: [] for method in DIRECTIONALITY_METHOD_ORDER
}
directionality_regulator_corr = {
    method: [] for method in DIRECTIONALITY_METHOD_ORDER
}
directionality_selected_mi_records = []

for slice_index, (adata_sub, graph_sub, factor_sub) in enumerate(
    zip(adata_list, SpiderNet_data_pyg_list, Factor_envir_list)
):
    slice_label, metadata_key, metadata_value = dir_get_slice_label_and_metadata(
        adata_sub,
        slice_index,
    )
    print(
        f"Direction-reversal coupling | {slice_label} "
        f"({slice_index + 1}/{len(adata_list)})"
    )

    edge_index_sub = dir_edge_index_to_numpy(graph_sub["edge_index"])
    factor_sub = dir_to_numpy_2d(
        factor_sub,
        name=f"Factor_envir_list[{slice_index}]",
    )
    n_cells = int(adata_sub.n_obs)

    if edge_index_sub.shape[0] != factor_sub.shape[0]:
        raise ValueError(
            f"Slice {slice_index + 1}: edge count and MI activity rows differ: "
            f"{edge_index_sub.shape[0]} vs {factor_sub.shape[0]}"
        )

    target_df, target_pair_names, _ = cell_by_pair_mean_genesets(
        adata_sub,
        targets_by_LR_unique,
        use_layer=None,
        fill_value=np.nan,
    )
    regulator_df, regulator_pair_names, _ = cell_by_pair_mean_genesets(
        adata_sub,
        regulators_by_LR_unique,
        use_layer=None,
        fill_value=np.nan,
    )

    agg_original = dir_aggregate_sender_receiver(
        factor_sub,
        edge_index_sub,
        n_cells,
    )
    agg_reverse = {
        "sender": agg_original["receiver"],
        "receiver": agg_original["sender"],
    }

    # Receiver targets: select MI using original SpiderNet receiver coupling.
    target_original_matrix = dir_compute_pair_mi_spearman_matrix(
        target_df,
        agg_original["receiver"],
    )
    target_reverse_matrix = dir_compute_pair_mi_spearman_matrix(
        target_df,
        agg_reverse["receiver"],
    )
    target_selected_mi = dir_select_reference_argmax_mi(target_original_matrix)

    directionality_target_corr["SpiderNet"].append(
        dir_extract_selected_mi_correlations(
            target_original_matrix,
            target_pair_names,
            target_selected_mi,
            slice_label,
        )
    )
    directionality_target_corr[REVERSE_DIRECTION_METHOD].append(
        dir_extract_selected_mi_correlations(
            target_reverse_matrix,
            target_pair_names,
            target_selected_mi,
            slice_label,
        )
    )
    directionality_selected_mi_records.extend(
        dir_build_lr_mi_selection_records(
            target_pair_names,
            target_selected_mi,
            target_original_matrix,
            target_reverse_matrix,
            slice_label,
            metadata_key,
            metadata_value,
            "Receiver targets",
        )
    )

    # Sender regulators: select MI using original SpiderNet sender coupling.
    regulator_original_matrix = dir_compute_pair_mi_spearman_matrix(
        regulator_df,
        agg_original["sender"],
    )
    regulator_reverse_matrix = dir_compute_pair_mi_spearman_matrix(
        regulator_df,
        agg_reverse["sender"],
    )
    regulator_selected_mi = dir_select_reference_argmax_mi(
        regulator_original_matrix
    )

    directionality_regulator_corr["SpiderNet"].append(
        dir_extract_selected_mi_correlations(
            regulator_original_matrix,
            regulator_pair_names,
            regulator_selected_mi,
            slice_label,
        )
    )
    directionality_regulator_corr[REVERSE_DIRECTION_METHOD].append(
        dir_extract_selected_mi_correlations(
            regulator_reverse_matrix,
            regulator_pair_names,
            regulator_selected_mi,
            slice_label,
        )
    )
    directionality_selected_mi_records.extend(
        dir_build_lr_mi_selection_records(
            regulator_pair_names,
            regulator_selected_mi,
            regulator_original_matrix,
            regulator_reverse_matrix,
            slice_label,
            metadata_key,
            metadata_value,
            "Sender regulators",
        )
    )


directionality_target_corr_frames = {
    method: pd.concat(frames, axis=0)
    for method, frames in directionality_target_corr.items()
}
directionality_regulator_corr_frames = {
    method: pd.concat(frames, axis=0)
    for method, frames in directionality_regulator_corr.items()
}
directionality_selected_mi_df = pd.DataFrame(
    directionality_selected_mi_records
)

for method, df in directionality_target_corr_frames.items():
    safe_method = method.replace(" ", "_").replace("-", "_")
    df.to_csv(
        os.path.join(
            directionality_outdir,
            f"Analysis2_receiver_target_corr_{safe_method}.csv",
        )
    )
for method, df in directionality_regulator_corr_frames.items():
    safe_method = method.replace(" ", "_").replace("-", "_")
    df.to_csv(
        os.path.join(
            directionality_outdir,
            f"Analysis2_sender_regulator_corr_{safe_method}.csv",
        )
    )

directionality_selected_mi_path = os.path.join(
    directionality_outdir,
    "Analysis2_original_SpiderNet_selected_MI_by_slice_LR.csv",
)
directionality_selected_mi_df.to_csv(
    directionality_selected_mi_path,
    index=False,
)

directionality_coupling_long_df = pd.concat(
    [
        dir_corr_frames_to_long(
            directionality_target_corr_frames,
            "Receiver targets",
        ),
        dir_corr_frames_to_long(
            directionality_regulator_corr_frames,
            "Sender regulators",
        ),
    ],
    ignore_index=True,
)

directionality_coupling_summary_records = []
for feature_type in ["Receiver targets", "Sender regulators"]:
    sub_ft = directionality_coupling_long_df[
        directionality_coupling_long_df["FeatureType"] == feature_type
    ]
    for representative_lr in sub_ft["RepresentativeLR"].unique():
        for method in DIRECTIONALITY_METHOD_ORDER:
            vals = sub_ft.loc[
                (sub_ft["RepresentativeLR"] == representative_lr)
                & (sub_ft["Method"] == method),
                "Correlation",
            ].to_numpy(dtype=float)
            mean_v, ci_low, ci_high, n = mean_ci95(vals)
            directionality_coupling_summary_records.append({
                "FeatureType": feature_type,
                "RepresentativeLR": representative_lr,
                "Method": method,
                "mean": mean_v,
                "ci95_low": ci_low,
                "ci95_high": ci_high,
                "n_slices": n,
            })

directionality_coupling_summary_df = pd.DataFrame(
    directionality_coupling_summary_records
)
directionality_coupling_long_path = os.path.join(
    directionality_outdir,
    "Analysis2_regulatory_coupling_long.csv",
)
directionality_coupling_summary_path = os.path.join(
    directionality_outdir,
    "Analysis2_regulatory_coupling_summary.csv",
)
directionality_coupling_long_df.to_csv(
    directionality_coupling_long_path,
    index=False,
)
directionality_coupling_summary_df.to_csv(
    directionality_coupling_summary_path,
    index=False,
)

print("Saved:", directionality_selected_mi_path)
print("Saved:", directionality_coupling_long_path)
print("Saved:", directionality_coupling_summary_path)
# Intermediate table display omitted.


## Prepare combined regulatory-coupling tables

Reuse the comparator scores calculated above. Append the reverse-direction
control at the MI selected by the original direction, last in the method order.


In [ ]:
# Combine the original benchmark methods with the reverse-direction control.
directionality_reverse_only_long_df = directionality_coupling_long_df[
    directionality_coupling_long_df["Method"] == REVERSE_DIRECTION_METHOD
][[
    "Slice",
    "RepresentativeLR",
    "Correlation",
    "Method",
    "FeatureType",
]].copy()

benchmark1_with_reverse_long_df = pd.concat(
    [
        benchmark1_long_df.copy(),
        directionality_reverse_only_long_df,
    ],
    axis=0,
    ignore_index=True,
)

benchmark1_with_reverse_method_order = [
    method
    for method in benchmark1_method_order
    if method != REVERSE_DIRECTION_METHOD
] + [REVERSE_DIRECTION_METHOD]

benchmark1_with_reverse_long_path = os.path.join(
    directionality_outdir,
    "Analysis2_Benchmark1_with_reverse_direction_long.csv",
)
benchmark1_with_reverse_long_df.insert(0, "Dataset", DATASET)
benchmark1_with_reverse_long_df.to_csv(
    benchmark1_with_reverse_long_path,
    index=False,
)

print("Method order:", benchmark1_with_reverse_method_order)
print("saved:", benchmark1_with_reverse_long_path)


## Combined two-study displays

Complete the second study, then draw both displays from the per-study CSVs.
Combined figures are saved under `output/CCC_Coupling_benchmark_with_directionality/`.
Use `python run_benchmarks.py --stage coupling --plot-only` to redraw them
without repeating the analyses.


In [ ]:
# Complete the second study, then render the shared saved tables.
if RUN_ALL_STUDIES:
    from ccc_directionality_two_studies import (
        plot_two_study_displays,
        run_additional_studies,
    )

    run_additional_studies(
        notebook_path=NOTEBOOK_PATH,
        datasets=DATASETS_TO_RUN,
        current_dataset=DATASET,
    )
    combined_output_dir = (
        NOTEBOOK_PATH.parent
        / "output"
        / "CCC_Coupling_benchmark_with_directionality"
    )
    (
        benchmark1_with_reverse_combined_png,
        benchmark1_with_reverse_combined_pdf,
        directionality_boxplot_combined_png,
        directionality_boxplot_combined_pdf,
    ) = plot_two_study_displays(
        DATASET_CONFIG, DATASETS_TO_RUN, METHOD_COLORS, combined_output_dir,
    )

    print("Combined study order:", DATASETS_TO_RUN)
    print("saved:", benchmark1_with_reverse_combined_png)
    print("saved:", benchmark1_with_reverse_combined_pdf)
    print("saved:", directionality_boxplot_combined_png)
    print("saved:", directionality_boxplot_combined_pdf)
else:
    print(f"Single-study subprocess completed: {DATASET}")
